<a href="https://colab.research.google.com/github/allisona-ctrl/tesis-ood-galaxias/blob/main/pipeline_ood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#=========================================================================
# IMPORTS Y CONFIGURACIÓN
#=========================================================================
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import gc
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, precision_score, accuracy_score
from numpy.linalg import inv
from IPython.display import Image, display
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")
#=========================================================================
# INGESTA Y VISUALIZACIÓN INICIAL
#=========================================================================
from IPython.display import Image, display
ruta_archivo = "/content/drive/MyDrive/Tesis/Galaxy10_DECals_NoDuplicated.h5"


with h5py.File(ruta_archivo, 'r') as f:
    for key in f.keys():
        print(f"{key}: {f[key].shape}")


    imagen = f['images'][0]
    etiqueta = f['ans'][0]


    plt.figure(figsize=(5,5))
    plt.imshow(imagen)
    plt.title(f"Clase {etiqueta}")
    plt.axis('off')
    plt.show()


    imagenes = f['images'][:]
    etiquetas = f['ans'][:]


plt.figure(figsize=(15, 6))
for i in range(10):
    idx = np.where(etiquetas == i)[0][0]
    plt.subplot(2, 5, i+1)
    plt.imshow(imagenes[idx])
    plt.title(f"Clase {i}")
    plt.axis('off')
plt.tight_layout()
plt.show()


del imagenes, etiquetas
gc.collect()


#=========================================================================
# PREPROCESAMIENTO Y DEFINICIÓN DEL ESCENARIO OOD
#=========================================================================
clases_ood = [8, 9]
clases_id = [0, 1, 2, 3, 4, 5, 6, 7]


with h5py.File(ruta_archivo, 'r') as f:
    etiquetas = f['ans'][:]
    mascara_id = np.isin(etiquetas, clases_id)
    mascara_ood = np.isin(etiquetas, clases_ood)
    indices_id = np.where(mascara_id)[0]
    indices_ood = np.where(mascara_ood)[0]


    np.random.seed(42)
    indices_id_sample = np.random.choice(indices_id, size=min(12000, len(indices_id)), replace=False)
    indices_ood_sample = np.random.choice(indices_ood, size=min(3000, len(indices_ood)), replace=False)


    X_id = f['images'][:][indices_id_sample]
    y_id = etiquetas[indices_id_sample]
    X_ood = f['images'][:][indices_ood_sample]
    y_ood = etiquetas[indices_ood_sample]


X_train, X_temp, y_train, y_temp = train_test_split(X_id, y_id, test_size=0.3, random_state=42, stratify=y_id)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)


carpeta_guardado = "/content/drive/MyDrive/Tesis/datos_procesados"
os.makedirs(carpeta_guardado, exist_ok=True)


np.savez_compressed(f"{carpeta_guardado}/train.npz", X=X_train.astype(np.uint8), y=y_train)
np.savez_compressed(f"{carpeta_guardado}/val.npz", X=X_val.astype(np.uint8), y=y_val)
np.savez_compressed(f"{carpeta_guardado}/test_id.npz", X=X_test.astype(np.uint8), y=y_test)
np.savez_compressed(f"{carpeta_guardado}/test_ood.npz", X=X_ood.astype(np.uint8), y=y_ood)


del X_id, y_id, X_ood, y_ood, X_train, X_temp, X_val, X_test
gc.collect()


#=========================================================================
# ARQUITECTURA Y DATA LOADERS
#=========================================================================
class GalaxyDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform


    def __len__(self):
        return len(self.X)


    def __getitem__(self, idx):
        img = torch.tensor(self.X[idx]).permute(2, 0, 1).float() / 255.0
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.y[idx], dtype=torch.long)


transformaciones = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


transformaciones_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def crear_modelo(num_clases):
    modelo = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    for param in modelo.parameters():
        param.requires_grad = False
    num_features = modelo.fc.in_features
    modelo.fc = nn.Linear(num_features, num_clases)
    return modelo.to(device)


ruta_datos = "/content/drive/MyDrive/Tesis/datos_procesados"
train_data = np.load(f"{ruta_datos}/train.npz")
val_data = np.load(f"{ruta_datos}/val.npz")
test_id_data = np.load(f"{ruta_datos}/test_id.npz")
test_ood_data = np.load(f"{ruta_datos}/test_ood.npz")


batch_size = 32
train_loader = DataLoader(GalaxyDataset(train_data['X'], train_data['y'], transformaciones), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(GalaxyDataset(val_data['X'], val_data['y'], transformaciones), batch_size=batch_size, shuffle=False)
test_id_loader = DataLoader(GalaxyDataset(test_id_data['X'], test_id_data['y'], transformaciones), batch_size=batch_size, shuffle=False)
test_ood_loader = DataLoader(GalaxyDataset(test_ood_data['X'], test_ood_data['y'], transformaciones), batch_size=batch_size, shuffle=False)
train_loader_aug = DataLoader(GalaxyDataset(train_data['X'], train_data['y'], transformaciones_train), batch_size=batch_size, shuffle=True)
#=========================================================================
# ENTRENAMIENTO Y FINE-TUNING
#=========================================================================
import torch.optim as optim


def calcular_accuracy(loader, model):
    model.eval()
    correctas = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correctas += (preds == labels).sum().item()
            total += labels.size(0)
    return correctas / total




historial_train_loss = []
historial_val_acc_fase1 = []
historial_val_acc_fase2 = []


modelo_base = crear_modelo(num_clases=8)
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_base.fc.parameters(), lr=0.001)


for epoch in range(5):
    modelo_base.train()
    loss_epoch = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizador.zero_grad()
        outputs = modelo_base(imgs)
        loss = criterio(outputs, labels)
        loss.backward()
        optimizador.step()
        loss_epoch += loss.item()


    val_acc = calcular_accuracy(val_loader, modelo_base)
    historial_train_loss.append(loss_epoch / len(train_loader))
    historial_val_acc_fase1.append(val_acc)
    print(f"[Fase 1] Época {epoch+1}/5 - Loss: {loss_epoch/len(train_loader):.4f} - Val Acc: {val_acc*100:.2f}%")
    torch.save(modelo_base.state_dict(), "/content/drive/MyDrive/Tesis/modelo_baseline_softmax.pth")




modelo_base = crear_modelo(num_clases=8)
for param in modelo_base.parameters():
    param.requires_grad = True


optimizador = optim.Adam(modelo_base.parameters(), lr=0.0001)
criterio = nn.CrossEntropyLoss()
mejor_val_acc = 0.0


for epoch in range(5):
    modelo_base.train()
    loss_epoch = 0.0
    for imgs, labels in train_loader_aug:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizador.zero_grad()
        outputs = modelo_base(imgs)
        loss = criterio(outputs, labels)
        loss.backward()
        optimizador.step()
        loss_epoch += loss.item()


    val_acc = calcular_accuracy(val_loader, modelo_base)
    historial_train_loss.append(loss_epoch / len(train_loader))
    historial_val_acc_fase2.append(val_acc)
    print(f"[Fase 2] Época {epoch+1}/5 - Loss: {loss_epoch/len(train_loader):.4f} - Val Acc: {val_acc*100:.2f}%")
    if val_acc > mejor_val_acc:
        mejor_val_acc = val_acc
        torch.save(modelo_base.state_dict(), "/content/drive/MyDrive/Tesis/modelo_baseline_finetuned.pth")


gc.collect()
torch.cuda.empty_cache()


#=========================================================================
# EVALUACIÓN OOD (ISOMAXPLUS, MC DROPOUT, MAHALANOBIS, ENERGY)
#=========================================================================
modelo_base.load_state_dict(torch.load("/content/drive/MyDrive/Tesis/modelo_baseline_finetuned.pth"))
modelo_base.eval()


def obtener_predicciones_y_confianzas(loader, model):
    todas_preds, todas_etiquetas, todas_confianzas = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            probabilidades = torch.softmax(outputs, dim=1)
            confianzas, preds = torch.max(probabilidades, 1)
            todas_preds.extend(preds.cpu().numpy())
            todas_etiquetas.extend(labels.cpu().numpy())
            todas_confianzas.extend(confianzas.cpu().numpy())
    return np.array(todas_preds), np.array(todas_etiquetas), np.array(todas_confianzas)


def extraer_caracteristicas(loader, model):
    model.eval()
    features_list, labels_list = [], []
    original_fc = model.fc
    model.fc = torch.nn.Identity()
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            features_list.append(model(imgs).cpu())
            labels_list.append(labels)
    model.fc = original_fc
    return torch.cat(features_list), torch.cat(labels_list)




preds_id, etiquetas_id, conf_id = obtener_predicciones_y_confianzas(test_id_loader, modelo_base)
preds_ood, etiquetas_ood, conf_ood = obtener_predicciones_y_confianzas(test_ood_loader, modelo_base)




features_train, labels_train = extraer_caracteristicas(train_loader, modelo_base)
prototipos = {}
for clase in range(8):
    mask = labels_train == clase
    prototipos[clase] = features_train[mask].mean(dim=0)
tensor_prototipos = torch.stack([prototipos[c] for c in range(8)]).to(device)


#─────────────────────────────────────────────────────────────────────────
# ISOMAXPLUS (Score de Distancia a Prototipos)
#─────────────────────────────────────────────────────────────────────────
features_val, _ = extraer_caracteristicas(val_loader, modelo_base)
dist_val = torch.cdist(features_val.to(device), tensor_prototipos)
min_dist_val = dist_val.min(dim=1)[0].cpu().numpy()
umbral_isomax = np.percentile(min_dist_val, 95)


features_id, _ = extraer_caracteristicas(test_id_loader, modelo_base)
features_ood, _ = extraer_caracteristicas(test_ood_loader, modelo_base)
dist_id = torch.cdist(features_id.to(device), tensor_prototipos)
dist_ood = torch.cdist(features_ood.to(device), tensor_prototipos)
min_dist_id = dist_id.min(dim=1)[0].cpu().numpy()
min_dist_ood = dist_ood.min(dim=1)[0].cpu().numpy()


detecciones_isomax = min_dist_ood > umbral_isomax
tasa_isomax = (np.sum(detecciones_isomax) / len(min_dist_ood)) * 100
fp_isomax = np.sum(min_dist_id > umbral_isomax)  # ← Falsos positivos ID


#─────────────────────────────────────────────────────────────────────────
# MC DROPOUT (Framework Híbrido)
#─────────────────────────────────────────────────────────────────────────
def predecir_con_incertidumbre_real(loader, model, T=20):
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()
    varianzas, distancias_minimas = [], []
    with torch.no_grad():
        original_fc = model.fc
        model.fc = nn.Identity()
        for imgs, _ in loader:
            imgs = imgs.to(device)
            features_pasadas = []
            for _ in range(T):
                features_pasadas.append(model(imgs))
            features_pasadas = torch.stack(features_pasadas)
            media_features = features_pasadas.mean(dim=0)
            varianza_features = features_pasadas.var(dim=0).mean(dim=1)
            dist = torch.cdist(media_features, tensor_prototipos)
            min_dist = dist.min(dim=1)[0]
            varianzas.extend(varianza_features.cpu().numpy())
            distancias_minimas.extend(min_dist.cpu().numpy())
        model.fc = original_fc
    return np.array(varianzas), np.array(distancias_minimas)


class ResNetConDropout(nn.Module):
    def __init__(self, modelo_base, p_dropout=0.5):
        super().__init__()
        self.backbone = nn.Sequential(*list(modelo_base.children())[:-1])
        self.dropout = nn.Dropout(p=p_dropout)
        self.fc = modelo_base.fc
    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


modelo_dropout = ResNetConDropout(modelo_base, p_dropout=0.5).to(device)
for param in modelo_dropout.backbone.parameters():
    param.requires_grad = False


optimizador = optim.Adam(modelo_dropout.fc.parameters(), lr=0.0001)
for epoch in range(2):
    modelo_dropout.train()
    for imgs, labels in train_loader_aug:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizador.zero_grad()
        outputs = modelo_dropout(imgs)
        loss = criterio(outputs, labels)
        loss.backward()
        optimizador.step()


torch.save(modelo_dropout.state_dict(), "/content/drive/MyDrive/Tesis/modelo_con_dropout.pth")




var_val, dist_val_hib = predecir_con_incertidumbre_real(val_loader, modelo_dropout)
scaler = MinMaxScaler()
scaler.fit(np.column_stack([var_val, dist_val_hib]))
score_val = scaler.transform(np.column_stack([var_val, dist_val_hib])).mean(axis=1)
umbral_hibrido = np.percentile(score_val, 95)




var_id, dist_id_hib = predecir_con_incertidumbre_real(test_id_loader, modelo_dropout)
var_ood, dist_ood_hib = predecir_con_incertidumbre_real(test_ood_loader, modelo_dropout)
score_id = scaler.transform(np.column_stack([var_id, dist_id_hib])).mean(axis=1)
score_ood = scaler.transform(np.column_stack([var_ood, dist_ood_hib])).mean(axis=1)


detecciones_hibrido = score_ood > umbral_hibrido
tasa_hibrido = (np.sum(detecciones_hibrido) / len(score_ood)) * 100
fp_hibrido = np.sum(score_id > umbral_hibrido)


#─────────────────────────────────────────────────────────────────────────
# 3. MAHALANOBIS
#─────────────────────────────────────────────────────────────────────────
def obtener_features(loader, model):
    model.eval()
    features_list = []
    original_fc = model.fc
    model.fc = torch.nn.Identity()
    with torch.no_grad():
        for imgs, _ in loader:
            features_list.append(model(imgs.to(device)).cpu())
    model.fc = original_fc
    return torch.cat(features_list)


features_train = obtener_features(train_loader, modelo_base).numpy()
features_train_np = features_train[:5000]
media_global = features_train_np.mean(axis=0)
cov = np.cov(features_train_np, rowvar=False)
cov_inv = inv(cov + np.eye(cov.shape[0]) * 1e-6)


def mahalanobis_dist(features):
    diff = features - media_global
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))




features_val_maha = obtener_features(val_loader, modelo_base).numpy()
maha_val = mahalanobis_dist(features_val_maha)
umbral_maha = np.percentile(maha_val, 95)




features_id_final = obtener_features(test_id_loader, modelo_base).numpy()
features_ood_final = obtener_features(test_ood_loader, modelo_base).numpy()
maha_id = mahalanobis_dist(features_id_final)
maha_ood = mahalanobis_dist(features_ood_final)


detecciones_maha = maha_ood > umbral_maha
tasa_maha = (np.sum(detecciones_maha) / len(maha_ood)) * 100
fp_maha = np.sum(maha_id > umbral_maha)


np.savez_compressed('/content/drive/MyDrive/Tesis/mahalanobis_params.npz',
                    media_global=media_global,
                    cov_inv=cov_inv,
                    umbral_maha=umbral_maha)


#─────────────────────────────────────────────────────────────────────────
# ENERGY SCORE
#─────────────────────────────────────────────────────────────────────────
def obtener_energias(loader, model):
    logits_list = []
    with torch.no_grad():
        for imgs, _ in loader:
            logits_list.append(model(imgs.to(device)).cpu())
    logits = torch.cat(logits_list)
    return -torch.logsumexp(logits, dim=1).numpy()




energy_val = obtener_energias(val_loader, modelo_base)
umbral_energy = np.percentile(energy_val, 5)




energy_id = obtener_energias(test_id_loader, modelo_base)
energy_ood = obtener_energias(test_ood_loader, modelo_base)


tasa_energy = (np.sum(energy_ood > umbral_energy) / len(energy_ood)) * 100
fp_energy = np.sum(energy_id > umbral_energy)


np.savez_compressed('/content/drive/MyDrive/Tesis/scores_ood.npz',
                    energy_id=energy_id, energy_ood=energy_ood,
                    maha_id=maha_id, maha_ood=maha_ood,
                    dist_id=min_dist_id, dist_ood=min_dist_ood,
                    var_id=var_id, var_ood=var_ood,
                    score_id=score_id, score_ood=score_ood)


gc.collect()
torch.cuda.empty_cache()
#=========================================================================
# GENERACIÓN DE FIGURAS Y TABLAS
#=========================================================================
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score


carpeta_tesis = "/content/drive/MyDrive/Tesis/figuras_tesis"
os.makedirs(carpeta_tesis, exist_ok=True)


#─────────────────────────────────────────────────────────────────────────
# Matriz de Confusión del Modelo Baseline (Test ID)
#─────────────────────────────────────────────────────────────────────────
matriz = confusion_matrix(etiquetas_id, preds_id)
plt.figure(figsize=(10, 8))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Matriz de Confusión - Modelo Baseline en Test ID', fontsize=14)
plt.xlabel('Predicción', fontsize=12)
plt.ylabel('Valor Real', fontsize=12)
plt.savefig(f"{carpeta_tesis}/fig09_matriz_confusion.png", dpi=300, bbox_inches='tight')
plt.close()


#─────────────────────────────────────────────────────────────────────────
# Evolución del accuracy de validación y la pérdida (Fine-Tuning)
#─────────────────────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 6))
epocas = range(1, len(historial_train_loss) + 1)


ax1.plot(epocas, historial_train_loss, 'o-', color='#C0392B', label='Pérdida (Train)')
ax1.set_xlabel('Épocas', fontsize=12)
ax1.set_ylabel('Pérdida', fontsize=12, color='#C0392B')
ax1.tick_params(axis='y', labelcolor='#C0392B')


ax2 = ax1.twinx()
val_acc_combinado = historial_val_acc_fase1 + historial_val_acc_fase2
ax2.plot(epocas, val_acc_combinado, 's-', color='#2E86C1', label='Accuracy (Val)')
ax2.set_ylabel('Accuracy', fontsize=12, color='#2E86C1')
ax2.tick_params(axis='y', labelcolor='#2E86C1')


plt.title('Evolución del Entrenamiento (Fine-Tuning)', fontsize=14)
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.grid(alpha=0.3)
plt.savefig(f"{carpeta_tesis}/fig10_evolucion_entrenamiento.png", dpi=300, bbox_inches='tight')
plt.close()


#─────────────────────────────────────────────────────────────────────────
# CARGA DE SCORES
#─────────────────────────────────────────────────────────────────────────
scores = np.load('/content/drive/MyDrive/Tesis/scores_ood.npz')


#─────────────────────────────────────────────────────────────────────────
# Comparación de distribuciones (IsoMaxPlus vs Mahalanobis)
#─────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))


ax1.hist(scores['dist_id'], bins=50, alpha=0.6, label='ID', color='blue', density=True)
ax1.hist(scores['dist_ood'], bins=50, alpha=0.6, label='OOD', color='red', density=True)
ax1.axvline(umbral_isomax, color='black', linestyle='--', label='Umbral 95%')
ax1.set_title(f'IsoMaxPlus (Distancia Euclidiana) - {tasa_isomax:.2f}%', fontsize=12)
ax1.set_xlabel('Distancia al Prototipo')
ax1.set_ylabel('Densidad')
ax1.legend()
ax1.grid(alpha=0.3)


ax2.hist(scores['maha_id'], bins=50, alpha=0.6, label='ID', color='blue', density=True)
ax2.hist(scores['maha_ood'], bins=50, alpha=0.6, label='OOD', color='red', density=True)
ax2.axvline(umbral_maha, color='black', linestyle='--', label='Umbral 95%')
ax2.set_title(f'Mahalanobis (Calibración Fina) - {tasa_maha:.2f}%', fontsize=12)
ax2.set_xlabel('Distancia de Mahalanobis')
ax2.set_ylabel('Densidad')
ax2.legend()
ax2.grid(alpha=0.3)


plt.suptitle('Comparación de Métodos de Detección Out-of-Distribution', fontsize=16)
plt.tight_layout()
plt.savefig(f"{carpeta_tesis}/fig11_histogramas_comparativos.png", dpi=300, bbox_inches='tight')
plt.close()


#─────────────────────────────────────────────────────────────────────────
# Distribución de Mahalanobis (figura final de la tesis)
#─────────────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.hist(scores['maha_id'], bins=60, alpha=0.6, label='ID (Galaxias Conocidas)', color='blue', density=True)
plt.hist(scores['maha_ood'], bins=60, alpha=0.6, label='OOD (Galaxias Desconocidas)', color='red', density=True)
plt.axvline(umbral_maha, color='black', linestyle='--', linewidth=2, label=f'Umbral 95% ID = {umbral_maha:.2f}')
plt.title('Distancia de Mahalanobis: Galaxias Conocidas vs Desconocidas', fontsize=14)
plt.xlabel('Distancia de Mahalanobis', fontsize=12)
plt.ylabel('Densidad', fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.savefig(f"{carpeta_tesis}/fig11b_mahalanobis.png", dpi=300, bbox_inches='tight')
plt.close()


#─────────────────────────────────────────────────────────────────────────
# Hiperparámetros (imagen)
#─────────────────────────────────────────────────────────────────────────
datos_tabla2 = {
    "Parámetro": ["Arquitectura", "Capas congeladas", "Optimizador", "Tasa de aprendizaje", "Función de pérdida", "Épocas", "Data Augmentation", "Batch size", "Accuracy de validación"],
    "Fase 1 (Feature Extraction)": ["ResNet-34", "Todas las convolucionales", "Adam", "0.001", "CrossEntropyLoss", "5", "No", "32", "41.61%"],
    "Fase 2 (Fine-Tuning)": ["ResNet-34", "Ninguna (descongeladas)", "Adam", "0.0001", "CrossEntropyLoss", "5", "RandomHorizontalFlip (p=0.5)", "32", "81.11%"]
}
df = pd.DataFrame(datos_tabla2)
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')
tabla = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center', colColours=['#2C3E50'] * 3)
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.2, 1.8)
for (i, j), cell in tabla.get_celld().items():
    if i == 0:
        cell.set_text_props(color='white', fontweight='bold')
    elif i % 2 == 0:
        cell.set_facecolor('#F2F2F2')
plt.title("Configuración de Hiperparámetros del Entrenamiento", fontsize=14, fontweight='bold', pad=20)
plt.savefig(f"{carpeta_tesis}/tabla_2_hiperparametros.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.close()


#─────────────────────────────────────────────────────────────────────────
# Rendimiento por clase (imagen)
#─────────────────────────────────────────────────────────────────────────
precision_por_clase = precision_score(etiquetas_id, preds_id, average=None, labels=range(8))
nombres_clases = [
    "Espiral Barrada",
    "Espiral No Barrada (Floja)",
    "Espiral No Barrada (Apretada)",
    "Elíptica Redonda",
    "Elíptica Alargada",
    "Borde con Bulbo",
    "Borde sin Bulbo",
    "Perturbada"
]
datos_tabla3 = {
    "Clase": [str(i) for i in range(8)],
    "Nombre": nombres_clases,
    "Precisión": [f"{p*100:.1f}%" for p in precision_por_clase]
}
df = pd.DataFrame(datos_tabla3)
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('tight')
ax.axis('off')
tabla = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center', colColours=['#2C3E50'] * 3)
tabla.auto_set_font_size(False)
tabla.set_fontsize(11)
tabla.scale(1.2, 1.8)
for (i, j), cell in tabla.get_celld().items():
    if i == 0:
        cell.set_text_props(color='white', fontweight='bold')
    elif i % 2 == 0:
        cell.set_facecolor('#F2F2F2')
plt.title("Rendimiento del Modelo Baseline por Clase Morfológica", fontsize=14, fontweight='bold', pad=20)
plt.savefig(f"{carpeta_tesis}/tabla_3_rendimiento_clases.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.close()


#─────────────────────────────────────────────────────────────────────────
# Comparación final de métodos
#─────────────────────────────────────────────────────────────────────────
datos = {
    "Método": ["SoftMax (Baseline)", "IsoMaxPlus", "MC Dropout (Híbrido)", "Mahalanobis", "Energy Score"],
    "Detección OOD (%)": ["0.0", f"{tasa_isomax:.2f}", f"{tasa_hibrido:.2f}", f"{tasa_maha:.2f}", f"{tasa_energy:.2f}"],
    "Falsos Positivos (ID)": ["-", f"{fp_isomax} / 1800", f"{fp_hibrido} / 1800", f"{fp_maha} / 1800", f"{fp_energy} / 1800"],
    "Conclusión": ["El modelo es ciego", "Detecta poco", "El ruido tapa la señal", "Óptimo equilibrio", "Demasiado agresivo"]
}
df = pd.DataFrame(datos)
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('tight')
ax.axis('off')
tabla = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center', colColours=['#2C3E50'] * 4)
tabla.auto_set_font_size(False)
tabla.set_fontsize(11)
tabla.scale(1.2, 1.8)
for (i, j), cell in tabla.get_celld().items():
    if i == 0:
        cell.set_text_props(color='white', fontweight='bold')
    if i == 4:
        cell.set_facecolor('#D4EDDA')
        cell.set_text_props(fontweight='bold')
plt.savefig(f"{carpeta_tesis}/tabla04_comparativa_final.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.close()
#=========================================================================
# SALIDA DEL PROGRAMA (RESUMEN DE RESULTADOS)
#=========================================================================
from sklearn.metrics import accuracy_score
import numpy as np


acc_id = accuracy_score(etiquetas_id, preds_id)
confianza_id = np.mean(conf_id) * 100
confianza_ood = np.mean(conf_ood) * 100


print("="*60)
print("RESUMEN DE RESULTADOS PARA LA TESIS")
print("="*60)


print(f"\n1. BASELINE (SOFTMAX)")
print(f"   - Accuracy en Test ID: {acc_id*100:.2f}%")
print(f"   - Confianza promedio en ID: {confianza_id:.2f}%")
print(f"   - Confianza promedio en OOD: {confianza_ood:.2f}%")


print(f"\n2. ISOMAXPLUS")
print(f"   - Detección OOD: {tasa_isomax:.2f}%")
print(f"   - Falsos Positivos: {fp_isomax} de {len(min_dist_id)}")


print(f"\n3. FRAMEWORK HÍBRIDO (MC DROPOUT)")
print(f"   - Detección OOD: {tasa_hibrido:.2f}%")
print(f"   - Falsos Positivos: {fp_hibrido} de {len(score_id)}")
print(f"   - Varianza promedio en ID: {np.mean(var_id):.4f}")
print(f"   - Varianza promedio en OOD: {np.mean(var_ood):.4f}")


print(f"\n4. MAHALANOBIS")
print(f"   - Detección OOD: {tasa_maha:.2f}%")
print(f"   - Falsos Positivos: {fp_maha} de {len(maha_id)}")
print(f"   - Umbral (Percentil 95 ID): {umbral_maha:.2f}")


print(f"\n5. ENERGY SCORE")
print(f"   - Detección OOD: {tasa_energy:.2f}%")
print(f"   - Falsos Positivos: {fp_energy} de {len(energy_id)}")
#=========================================================================
# BOOTSTRAP - INTERVALOS DE CONFIANZA AL 95%
#=========================================================================
N_BOOT = 1000
SEED_BOOT = 42




def bootstrap_ic(score_id, score_ood, umbral, n_boot=N_BOOT, seed=SEED_BOOT):
    rng = np.random.default_rng(seed)
    id_ood = score_id > umbral
    ood_ood = score_ood > umbral
    tpr_boot = np.zeros(n_boot)
    fpr_boot = np.zeros(n_boot)
    for i in range(n_boot):
        idx_id = rng.integers(0, len(score_id), len(score_id))
        idx_ood = rng.integers(0, len(score_ood), len(score_ood))
        fpr_boot[i] = id_ood[idx_id].mean() * 100
        tpr_boot[i] = ood_ood[idx_ood].mean() * 100
    return {
        "tpr": ood_ood.mean() * 100,
        "tpr_lo": np.percentile(tpr_boot, 2.5),
        "tpr_hi": np.percentile(tpr_boot, 97.5),
        "fpr": id_ood.mean() * 100,
        "fpr_lo": np.percentile(fpr_boot, 2.5),
        "fpr_hi": np.percentile(fpr_boot, 97.5),
    }




res_isomax = bootstrap_ic(min_dist_id, min_dist_ood, umbral_isomax)
res_hibrido = bootstrap_ic(score_id, score_ood, umbral_hibrido)
res_maha = bootstrap_ic(maha_id, maha_ood, umbral_maha)
res_energy = bootstrap_ic(energy_id, energy_ood, umbral_energy)




print("\n" + "=" * 60)
print("INTERVALOS DE CONFIANZA AL 95% (BOOTSTRAP, 1000 REMUESTREOS)")
print("=" * 60)


print("\nISOMAXPLUS")
print(f"   - TPR OOD: {res_isomax['tpr']:.2f}%  IC95% [{res_isomax['tpr_lo']:.2f}, {res_isomax['tpr_hi']:.2f}]")
print(f"   - FPR ID:  {res_isomax['fpr']:.2f}%  IC95% [{res_isomax['fpr_lo']:.2f}, {res_isomax['fpr_hi']:.2f}]")


print("\nFRAMEWORK HIBRIDO (MC DROPOUT)")
print(f"   - TPR OOD: {res_hibrido['tpr']:.2f}%  IC95% [{res_hibrido['tpr_lo']:.2f}, {res_hibrido['tpr_hi']:.2f}]")
print(f"   - FPR ID:  {res_hibrido['fpr']:.2f}%  IC95% [{res_hibrido['fpr_lo']:.2f}, {res_hibrido['fpr_hi']:.2f}]")


print("\nMAHALANOBIS")
print(f"   - TPR OOD: {res_maha['tpr']:.2f}%  IC95% [{res_maha['tpr_lo']:.2f}, {res_maha['tpr_hi']:.2f}]")
print(f"   - FPR ID:  {res_maha['fpr']:.2f}%  IC95% [{res_maha['fpr_lo']:.2f}, {res_maha['fpr_hi']:.2f}]")


print("\nENERGY SCORE")
print(f"   - TPR OOD: {res_energy['tpr']:.2f}%  IC95% [{res_energy['tpr_lo']:.2f}, {res_energy['tpr_hi']:.2f}]")
print(f"   - FPR ID:  {res_energy['fpr']:.2f}%  IC95% [{res_energy['fpr_lo']:.2f}, {res_energy['fpr_hi']:.2f}]")
